### Structuring Output and Validation

#### Setup

In [9]:
import openai
import json
from jupyter_base import OpenAIClient
client = OpenAIClient()


In [17]:
email_text = """
Hi, I'm Sarah Connor from Cyberdyne Systems. You can reach me at sarah.connor@cyberdynesystems.com
or call me at (555) 121-1210. Looking forward to discussing the skynet plan.
"""

prompt = f"""
Extract the contact information from this email:
{email_text}
Return JSON with these exact keys: name, company, email, phone
Return only the JSON object. No markdown or explanations.
"""

In [18]:
response = client.responses.create(model="gpt-4.1-mini", input=prompt)
response = response.output[0].content[0].text
print(response)

{"name":"Sarah Connor","company":"Cyberdyne Systems","email":"sarah.connor@cyberdynesystems.com","phone":"(555) 121-1210"}


### Production Pattern
- Define a Schema
- Prompt for JSON
- Parse to Python
- Validate key/values
- Verify quality

In [19]:
def parse_json(text):
    """Parse JSON from the API output, handling markdown fences."""
    clean = text.strip()
    # strip markdown fences if present
    if clean.startswith('```'):
        clean = clean.split('```')[1]
        if clean.startswith('json'):
            clean = clean[4:]
        clean = clean.strip()

    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        return None

In [20]:
print(parse_json(response))

{'name': 'Sarah Connor', 'company': 'Cyberdyne Systems', 'email': 'sarah.connor@cyberdynesystems.com', 'phone': '(555) 121-1210'}


### JSON Response Example

In [23]:
fewshot_json_prompt = """
   Analyze product reviews and return structured data.

   Examples:
   Review: Amazing quality, highly recommended.
   {"sentiment": "positive", "confidence": 0.95, "key_points": ["quality", "recommendation"]}
   Review: Terrible product, stopped working after a week.
   {"sentiment": "negative", "confidence": 0.90, "key_points": ["quality issues", "durability"]}
   Review: It's OK, nothing special.
   {"sentiment": "neutral", "confidence": 0.85, "key_points": ["average"]}

   Now analyze:
   Review: Great features, but a bit pricey.
"""
response = client.responses.create(model="gpt-4.1-mini", input=fewshot_json_prompt)
response = response.output[0].content[0].text
data = parse_json(response)
print(data)

{'sentiment': 'neutral', 'confidence': 0.8, 'key_points': ['features', 'price']}


### JSON Schemas

In [25]:
CONTACT_SCHEMA = {
    "name": "string",
    "company": "string",
    "email": "string",
    "phone": "string"
}

In [27]:
prompt = f"""
Extract the contact information from this email:
{email_text}
Return JSON with these exact keys: name, company, email, phone
Return only the JSON object. No markdown or explanations.
Each object will have the following schema:
{json.dumps(CONTACT_SCHEMA, indent=2)}
"""
response = client.responses.create(model="gpt-4.1-mini", input=prompt)
response = response.output[0].content[0].text
data = parse_json(response)
print(data)

{'name': 'Sarah Connor', 'company': 'Cyberdyne Systems', 'email': 'sarah.connor@cyberdynesystems.com', 'phone': '(555) 121-1210'}
